# ECON2041 Week 7 tutorial: what makes a Sydney Airbnb listing expensive?

### What you'll be able to do by the end

- Use a histogram and a scatterplot to describe prices before fitting a model
- Fit a line of nightly price on the number of guests, then read and interpret the $R^2$
- Turn room type into a dummy variable and add it as a second explanatory variable
- Say what it means to "hold the number of guests fixed"
- Predict which way a coefficient will differ before you run the regression

### Different types of cells

- 🟢 Read and run: these cells have pre-written code, you can simply run them, read and interpret the output
- ✏️ You write: your turn to try to write code or sometimes add a short written answer
- 🤔 You think: stop and think on your own before running the next cell or reading on
- 💬 Discuss: talk with your neighbors before typing or running
- ⭐ Optional: extra exercise if you have time

## Setup

Our usual setup block, the same one as week 6, with `ols` from `statsmodels` included.

In [ ]:
# 🟢 Read and run: our standard ECON2041 setup block
import numpy as np               # numerical tools (nicknamed np)
import pandas as pd              # data tools (nicknamed pd)
import matplotlib.pyplot as plt  # plotting tools (nicknamed plt)
import seaborn as sns            # statistical charts (nicknamed sns)
from statsmodels.formula.api import ols  # ordinary least squares regression

# Keep scalar output plain under NumPy 2 (0.5, not np.float64(0.5)).
if np.lib.NumpyVersion(np.__version__) >= "2.0.0":
    np.set_printoptions(legacy="1.25")

DATA = "https://emiliatjernstrom.com/econ2041/data"   # Unit datasets live at this web address

print("Setup done!")

## The question

We'll work with Sydney Airbnb listings priced between $30 and $2,000 a night on 16 June 2026, 12,419 in total, and try to answer the following question:

> What makes a Sydney Airbnb listing expensive?

Source: [Inside Airbnb](https://insideairbnb.com/), Sydney, snapshot of 16 June 2026, CC BY 4.0.

🤔 Before you look at the columns, what characteristics do you think affect a listing's nightly price?

Write down at least three possibilities. For each one, say which direction you expect the association with price to go.

## The data

The file contains nine variables:

- `price`: nightly price in Australian dollars
- `accommodates`: number of guests the listing sleeps
- `bedrooms`: number of bedrooms
- `room_type`: entire home or apartment, private room, shared room, or hotel room
- `review_scores_rating`: average guest rating, from 1 to 5
- `neighbourhood`: the listing's area within Sydney
- `number_of_reviews`: number of reviews the listing has received
- `latitude` and `longitude`: coordinates that locate the listing

In [ ]:
# 🟢 Read and run: load sydney-airbnb-teaching.csv into a dataframe called airbnb and look at the first few rows
airbnb = pd.read_csv(f"{DATA}/sydney-airbnb-teaching.csv")

airbnb.head()

In [ ]:
# 🟢 Read and run: summary statistics for the nightly price
airbnb["price"].describe().round(1)

In [ ]:
# 🟢 Read and run: draw a histogram of nightly prices
sns.histplot(data=airbnb, x="price", bins=40, color="purple")
plt.xlabel("Nightly price (AUD)")
plt.ylabel("Number of listings")
plt.show()

🤔 Look at the histogram and the summary statistics together:

- Where are most listings concentrated, and which tail is longer?
- Which is larger, the mean or the median? What about the shape makes that happen?
- Which number better describes the price of a typical listing here: the mean or the median? Why?

## See the relationship

You suggested several characteristics that might affect price. We will begin with `accommodates`, the number of guests a listing sleeps.
First draw only the listings. We will add a fitted line after we run the regression.

### Question 1: how does price vary with the number of guests?

In [ ]:
# ✏️ You fill in: the code below is complete except for the two blanks.
# Fill them in, then remove the leading # from each line and run
# (in Colab: select the lines and press Ctrl+/ to uncomment them all at once).

# sns.scatterplot(data=airbnb,                      # the dataframe we just loaded
#                 x=________,                       # the explanatory variable, in quotes
#                 y=________,                       # the outcome, in quotes
#                 s=8, alpha=0.2, color="purple")  # small see-through dots
# plt.xlabel("Guests the listing sleeps")
# plt.ylabel("Nightly price (AUD)")
# plt.show()

🤔 Look at the picture:

- As you move right, toward listings that sleep more guests, do the dots drift up, down, or neither?
- Pick one column, say listings that sleep 4 guests, and scan up and down: how much do prices vary among listings that sleep the same number of guests?

## Fit the line and ask how well it fits

### Question 2: what line does least squares choose?

The formula follows the same pattern as last week: the outcome to the left of `~`, the explanatory variable to the right.

In [ ]:
# ✏️ You write: fit price on accommodates with ordinary least squares, store the result as model,
# and show the two estimated coefficients rounded to 1 decimal
# Hint: ols("outcome ~ explanatory", data=airbnb).fit() fits the line;
#       model.params reports the coefficients, and .round(1) rounds them

In [ ]:
# 🟢 Read and run: now add the least-squares line to the scatterplot

sns.regplot(data=airbnb, x="accommodates", y="price", ci=None,
            scatter_kws={"s": 8, "alpha": 0.2, "color": "purple"}, line_kws={"color": "orange"})
plt.xlabel("Guests the listing sleeps")
plt.ylabel("Nightly price (AUD)")
plt.show()

🤔 Write out the fitted line, $\widehat{\text{price}} = \hat{\beta}_0 + \hat{\beta}_1 \times \text{accommodates}$, with the two estimates filled in.

- What does the slope mean here?
- What does the intercept describe here? Check the smallest value of `accommodates` in the file before you answer, then compare with last week's intercept at `escs` $= 0$, which described plenty of real students.

### Question 3: how much of the variation in price does the line explain?

The week 6 Essential Concepts recordings split the total variation in the outcome into the part the line explains and the part left in the residuals:

$$R^2 = \frac{\text{ESS}}{\text{TSS}} = 1 - \frac{\text{RSS}}{\text{TSS}}$$

$R^2$ is the share of the variation in `price` that the fitted line explains, a number between 0 and 1.
The fitted model already has it stored under the name `rsquared`.

In [ ]:
# ✏️ You write: show the R squared of model, rounded to 2 decimals
# Hint: model.rsquared is a single number, and round(model.rsquared, 2) rounds it

✏️ Complete the sentence: the number of guests a listing sleeps explains about ________% of the variation in nightly price across listings.

🤔 Then think about the rest:

- What kinds of differences between listings could account for the other part of the variation?
- Suppose $R^2$ were 0.9. Would that make the number of guests the *cause* of the price?

## Add room type as a second explanatory variable

Room type gives us a different kind of explanatory variable.
`room_type` is a word, and a regression needs a number.
The week 6 live lecture turned unemployment into a dummy variable, `high_ue`, with `astype(int)`; we do the same with room type.

### Question 4: how much more does an entire home cost?

In [ ]:
# 🟢 Read and run: how many listings of each room type are in the file?
airbnb["room_type"].value_counts()

In [ ]:
# ✏️ You write: create a new column called entire that is 1 when room_type is "Entire home/apt" and 0 otherwise,
# then show the mean of the new column rounded to 2 decimals
# Hint: (airbnb["room_type"] == "Entire home/apt") asks every row the question and returns True or False;
#       .astype(int) turns True into 1 and False into 0; the left of = stores the result in a new column;
#       the mean of a column of 0s and 1s is the share of 1s

In [ ]:
# ✏️ You write: calculate the mean nightly price for rooms (entire = 0) and entire homes (entire = 1)
# Store the two means as mean_price_by_room_type, then show them rounded to 1 decimal
# Hint: airbnb.groupby("entire")["price"].mean() computes the mean price within each group

In [ ]:
# 🟢 Read and run: compare the two group means in a bar chart
mean_price_by_room_type.rename(index={0: "Room", 1: "Entire home"}).plot(
    kind="bar", color=["gray", "purple"])
plt.xlabel("Room type")
plt.ylabel("Mean nightly price (AUD)")
plt.xticks(rotation=0)
plt.show()

🤔 Why a bar chart here, when question 1 used a scatterplot?

### Question 5: predict first, then run

Entire homes cost more, but they also sleep more guests.

In [ ]:
# 🟢 Read and run: compare mean guests for rooms (entire = 0) and entire homes (entire = 1)
airbnb.groupby("entire")["accommodates"].mean().round(1)

🤔 Before you run anything, write down your two guesses. One clue: entire homes sleep 4.6 guests on average, compared with 2.2 for rooms, so they differ in both room type and the number of guests. Guests already have a coefficient, so ask which of the two variables is currently getting credit for the other.

Now predict:

- Will the coefficient on `accommodates` be larger or smaller than 68.5? Why?
- Will the coefficient on `entire` be larger or smaller than the raw gap of $271? Why?

In [ ]:
# ✏️ You write: fit price on accommodates and entire together, store the result as multi,
# and show the coefficients rounded to 1 decimal
# Hint: put a + between the two explanatory variables on the right of ~

✏️ Complete the two sentences:

- Holding the number of guests fixed, an entire home is priced about $________ higher a night than a room, on average.
- Holding room type fixed, a listing that sleeps 1 more guest is priced about $________ higher a night, on average.

🤔 Which of your two guesses came true? Compare the new guests coefficient with 68.5, and the new `entire` coefficient with the raw gap of $271.

In [ ]:
# 🟢 Read and run: draw the two parallel fitted lines over the scatterplot
b = multi.params
guests = np.arange(1, airbnb["accommodates"].max() + 1)

sns.scatterplot(data=airbnb, x="accommodates", y="price", s=8, alpha=0.2, color="purple")
plt.plot(guests, b["Intercept"] + b["accommodates"] * guests, color="gray", label="Room")
plt.plot(guests, b["Intercept"] + b["entire"] + b["accommodates"] * guests, color="orange", label="Entire home")
plt.xlabel("Guests the listing sleeps")
plt.ylabel("Nightly price (AUD)")
plt.legend()
plt.show()

### ⭐ Question 6 (optional): two listings, 4 guests each

Last week, you first plugged values into the fitted equation by hand and then checked several predictions at once with `model.predict()`. We will use the same two-step approach with two explanatory variables.

🤔 Before you run any code, use the two fitted lines to predict the mean price for a listing that sleeps 4 guests:

- Room: $16.1 + 63.8 \times 4$
- Entire home: $135.0 + 63.8 \times 4$

Now check both predictions at once. As in week 6, we collect the explanatory-variable values in a small dataframe and hand that dataframe to `.predict()`: two rows in, two predicted mean prices out.

In [ ]:
# 🟢 Read and run: check the two by-hand predictions with multi.predict()

new_listings = pd.DataFrame({"accommodates": [4, 4], "entire": [1, 0]})
new_listings["predicted_price"] = multi.predict(new_listings)
new_listings.round(1)

💬 With your neighbors:

- In plain words, what comparison does "holding the number of guests fixed" describe? Which listings is the $119 comparing?
- Can we interpret the $119 as the causal effect of being an entire-home listing? Why or why not?
- Name two things that differ between entire homes and private rooms that this model does not hold fixed

### ⭐ Question 7 (optional): use review score as a different control

For a second interpretation exercise, leave room type out. Fit `price` on `accommodates` and `review_scores_rating` instead. The review score runs from 1 to 5.

In [ ]:
# ✏️ You write (optional): fit price on accommodates and review_scores_rating together,
# store the result as rating_model, and show the coefficients rounded to 1 decimal
# Hint: use the same formula pattern as question 5, with a + between the two explanatory variables

✏️ Complete the two sentences:

- Holding review score fixed, a listing that sleeps 1 more guest is priced about $________ higher a night, on average.
- Holding the number of guests fixed, a listing with a review score 1 point higher is priced about $________ higher a night, on average.

🤔 Two more questions:

- Is a 1-point difference in rating common in these data? Run `airbnb["review_scores_rating"].describe()` to check
- Can we interpret either coefficient as a causal effect? Why or why not?